# VHR Download

Pulls Vacation Home Rental / Short Term Rental permit data from the five Tahoe
Basin jurisdictions, normalizes it to a common schema, flags each record as
inside or outside the Basin, and joins the result to `Parcel_Master`.

| Step | What it does |
| --- | --- |
| 1. Download | Pulls each jurisdiction's REST service to GeoJSON |
| 2. Normalize | Maps each source to `apn / status / max_occupancy / jurisdiction` + a point |
| 3. Status crosswalk | Collapses ~20 local permit statuses to Active / Inactive / Pending |
| 4. Basin flag | Marks each record inside or outside the TRPA boundary |
| 5. Filter | Keeps only Active permits inside the Basin |
| 6. Save | Writes `VHR_Combined.shp` / `.geojson` |
| 7. Parcel join | Loads the staging FGDB and spatially joins to `Parcel_Master`, as points and as parcel polygons |

**Outputs**

- `<PROJECT_FOLDER>/GeoJSON/*.geojson` - one raw download per source
- `<PROJECT_FOLDER>/GeoJSON/VHR_Combined.shp` and `.geojson` - the combined layer
- `VHR_Staging.gdb/VHR_Combined` and `VHR_Staging.gdb/VHR_Parcel_Join` (points)
- `VHR_Staging.gdb/VHR_Parcel_Polygon` - the same records on the `Parcel_Master` polygon
- `vhr_types.csv` - every status/jurisdiction pair seen this run

Step 5 onwards covers Active in-Basin permits only. `vhr_types.csv` and the status
and Basin reports in steps 3 and 4 still describe every record downloaded.

**Before running**

- Set the `Password` environment variable if the arcgis SDK download fallback is needed.
- Step 3 reads `vhr_types_lookup.csv`, which is maintained by hand. When a
  jurisdiction introduces a new permit status, step 3a lists it and step 3b warns
  about it; add a `status_simple` value for it before trusting the output.


## Setup

Paths, service URLs, and the portal connection. This is the only cell that
should need editing between runs.

In [1]:
"""VHR Download - configuration.

Everything that changes between runs (paths, service URLs, credentials) lives in
this cell so nothing further down needs editing.
"""
import os
import re
import json
import warnings

import pandas as pd
import geopandas as gpd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from shapely.geometry import shape, Point, Polygon, LineString
from shapely.ops import unary_union

import arcpy
from arcgis.gis import GIS

# --- Paths --------------------------------------------------------------------
PROJECT_FOLDER = r"F:\GIS\PROJECTS\ResearchAnalysis\VHR"
WORKSPACE      = os.path.join(PROJECT_FOLDER, "Data", "VHR_Staging.gdb")
OUTPUT_DIR     = os.path.join(PROJECT_FOLDER, "GeoJSON")

# Status crosswalk CSVs live next to this notebook and are version controlled
VHR_TYPES_CSV        = os.path.abspath("vhr_types.csv")         # written by this notebook
VHR_TYPES_LOOKUP_CSV = os.path.abspath("vhr_types_lookup.csv")  # maintained by hand

# Washoe County's REST service currently rejects query operations, so an export
# downloaded from their open data portal is used as a fallback.
WASHOE_SHAPEFILE = r"C:\Users\amcclary\Downloads\ShortTermRentals\ShortTermRentals.shp"

# Enterprise geodatabase targets used by the final spatial join
PARCEL_MASTER = r"F:\GIS\DB_CONNECT\Vector.sde\SDE.Parcels\SDE.Parcel_Master"
PARCEL_VHR    = r"F:\GIS\DB_CONNECT\Collection.sde\SDE.Parcel\SDE.Parcel_VHR"
VHR_FC        = os.path.join(WORKSPACE, "VHR_Combined")
VHR_PARCEL_FC = os.path.join(WORKSPACE, "VHR_Parcel_Join")
VHR_PARCEL_POLY_FC = os.path.join(WORKSPACE, "VHR_Parcel_Polygon")

# --- Coordinate systems -------------------------------------------------------
WGS84         = "EPSG:4326"   # everything is published in WGS84
PROJECTED_CRS = "EPSG:26910"  # NAD83 / UTM 10N - metres, used for centroids

# --- Source services ----------------------------------------------------------
SERVICE_URLS = {
    "ElDorado_VHR_Permits": "https://gis.eldoradocounty.ca.gov/mapping/rest/services/VHR/Permits/FeatureServer/0",
    "CSLT_VHR":             "https://services2.arcgis.com/gWRYLIS16mKUskSO/ArcGIS/rest/services/VHR_Public_View/FeatureServer/0",
    "Washoe_VHR":           "https://wcgisweb.washoecounty.us/arcgis/rest/services/OpenData/OpenData/FeatureServer/147",
    "Douglas_VHR_Occupancy":"https://gisservices.douglasnv.us/server/rest/services/VHR_Occupancy/MapServer/0",
    "Placer_VHR":           "https://services6.arcgis.com/PArfeTGcwA9RGNzN/ArcGIS/rest/services/STR_Permitted_Properties_FY_24_25/FeatureServer/0",
}
BASIN_URL = "https://maps.trpa.org/server/rest/services/Boundaries/MapServer/4"

# --- Environment --------------------------------------------------------------
arcpy.env.workspace = WORKSPACE       # note: use the raw string above, not a literal
arcpy.env.overwriteOutput = True
os.makedirs(OUTPUT_DIR, exist_ok=True)

# The portal sign-in is only needed for the arcgis-SDK download fallback, so a
# missing password is a warning rather than a hard failure.
PORTAL_URL  = "https://maps.trpa.org/portal/"
PORTAL_USER = "TRPA_PORTAL_ADMIN"
_portal_pwd = os.environ.get("Password")
if _portal_pwd:
    gis = GIS(PORTAL_URL, PORTAL_USER, _portal_pwd)
    print(f"Signed in to {PORTAL_URL}")
else:
    gis = None
    warnings.warn(
        "Environment variable 'Password' is not set - continuing anonymously. "
        "The arcgis SDK download fallback will not work for secured services."
    )

print("Workspace :", WORKSPACE)
print("Output dir:", OUTPUT_DIR)

Signed in to https://maps.trpa.org/portal/
Workspace : F:\GIS\PROJECTS\ResearchAnalysis\VHR\Data\VHR_Staging.gdb
Output dir: F:\GIS\PROJECTS\ResearchAnalysis\VHR\GeoJSON


## Download helpers

One downloader used by every source. Jurisdiction services vary widely in what
they support, so it degrades through response-format, paging, and transport
fallbacks rather than assuming a modern hosted FeatureServer.

In [2]:
"""Generic ArcGIS REST downloader.

Jurisdiction services differ in what they support, so the downloader degrades
through a series of fallbacks:

    f=geojson  ->  f=json (Esri JSON)              (response format)
    resultOffset paging  ->  objectIds chunking    (paging strategy)
    raw REST  ->  arcgis Python SDK                (transport)

Every query asks for outSR=4326 so the Esri JSON path cannot silently return
coordinates in the service's native projection.
"""

# Session with automatic retry on transient server errors
_session = requests.Session()
_adapter = HTTPAdapter(max_retries=Retry(total=3, backoff_factor=1, status_forcelist=[500, 502, 503, 504]))
_session.mount("http://", _adapter)
_session.mount("https://", _adapter)


def safe_get(url, params, timeout=60):
    """GET with SSL fallback and ArcGIS error detection."""
    try:
        r = _session.get(url, params=params, timeout=timeout)
    except requests.exceptions.SSLError:
        warnings.warn(f"SSL error for {url}, retrying without verification")
        r = _session.get(url, params=params, timeout=timeout, verify=False)
    r.raise_for_status()
    data = r.json()
    if "error" in data:
        err = data["error"]
        raise RuntimeError(f"Service error {err.get('code')}: {err.get('message')} {err.get('details', '')}")
    return data


def try_get_server_token(service_url):
    """Request an IP-based anonymous token from a self-hosted ArcGIS Server."""
    parts = service_url.split("/")
    base = "/".join(parts[:3])
    for i, p in enumerate(parts):          # trim to the /arcgis (or /server) root
        if p.lower() in ("arcgis", "server"):
            base = "/".join(parts[: i + 1])
            break
    token_url = f"{base}/tokens/generateToken"
    payload = {"client": "requestip", "f": "json"}
    for verify in (True, False):           # prefer a verified request before falling back
        try:
            r = _session.post(token_url, data=payload, timeout=15, verify=verify)
            if r.status_code == 200:
                token = r.json().get("token")
                if token:
                    return token
        except Exception:
            continue
    return None


def get_service_info(service_url, token=None):
    params = {"f": "json"}
    if token:
        params["token"] = token
    return safe_get(service_url, params)


def get_oid_field(service_info):
    for field in service_info.get("fields", []):
        if field.get("type") == "esriFieldTypeOID":
            return field["name"]
    return "OBJECTID"


def esri_geom_to_shapely(esri_geom, geom_type):
    """Convert Esri JSON geometry to shapely; used when a service can't emit GeoJSON."""
    if not esri_geom:
        return None
    try:
        if "Point" in geom_type:
            x, y = esri_geom.get("x"), esri_geom.get("y")
            return Point(x, y) if x is not None and y is not None else None
        if "Polygon" in geom_type:
            polys = [Polygon(r) for r in esri_geom.get("rings", []) if len(r) >= 3]
            return unary_union(polys) if polys else None
        if "Polyline" in geom_type:
            lines = [LineString(p) for p in esri_geom.get("paths", []) if len(p) >= 2]
            return unary_union(lines) if lines else None
    except Exception:
        return None
    return None


def esri_response_to_geojson_features(data):
    """Convert an Esri JSON query response to GeoJSON-style features."""
    geom_type = data.get("geometryType", "")
    features = []
    for f in data.get("features", []):
        geom = esri_geom_to_shapely(f.get("geometry"), geom_type)
        features.append({
            "type": "Feature",
            "properties": f.get("attributes", {}),
            "geometry": geom.__geo_interface__ if geom else None,
        })
    return features


def _response_epsg(data, default=4326):
    sr = data.get("spatialReference") or {}
    return sr.get("latestWkid") or sr.get("wkid") or default


def query_page(service_url, params, use_geojson=True):
    """Query one page. Returns (features, epsg, use_geojson_next)."""
    params = {**params, "outSR": 4326}
    if use_geojson:
        try:
            data = safe_get(f"{service_url}/query", {**params, "f": "geojson"})
            return data.get("features", []), 4326, True
        except Exception as e:
            warnings.warn(f"GeoJSON query failed ({e}), falling back to Esri JSON")
    data = safe_get(f"{service_url}/query", {**params, "f": "json"})
    return esri_response_to_geojson_features(data), _response_epsg(data), False


def _sdk_features_to_geojson(sdk_features):
    """Convert arcgis SDK Feature objects to GeoJSON-style feature dicts."""
    result = []
    for f in sdk_features:
        geom = None
        if f.geometry:
            try:
                esri_dict = dict(f.geometry)
                if "x" in esri_dict:
                    gt = "esriGeometryPoint"
                elif "rings" in esri_dict:
                    gt = "esriGeometryPolygon"
                elif "paths" in esri_dict:
                    gt = "esriGeometryPolyline"
                else:
                    gt = ""
                s = esri_geom_to_shapely(esri_dict, gt)
                geom = s.__geo_interface__ if s else None
            except Exception:
                pass
        result.append({"type": "Feature", "properties": dict(f.attributes), "geometry": geom})
    return result


def download_via_arcgis_sdk(service_url, max_records, where):
    """Last-resort transport for services that reject raw REST queries."""
    from arcgis.features import FeatureLayer

    print("  Trying arcgis Python SDK fallback...")
    layer = FeatureLayer(service_url)
    all_sdk = []
    offset = 0
    while True:
        result = layer.query(
            where=where,
            out_fields="*",
            out_sr=4326,
            result_offset=offset,
            result_record_count=max_records,
        )
        batch = result.features
        if not batch:
            break
        all_sdk.extend(batch)
        print(f"  {len(all_sdk)} features (SDK)...")
        if not getattr(result, "exceededTransferLimit", False):
            break
        offset += max_records
    return _sdk_features_to_geojson(all_sdk)


def features_to_gdf(features, epsg=4326):
    """Build a GeoDataFrame from GeoJSON-style features, handling absent geometry."""
    props = [f.get("properties", {}) or {} for f in features]
    geoms = [shape(f["geometry"]) if f.get("geometry") else None for f in features]
    if any(g is not None for g in geoms):
        gdf = gpd.GeoDataFrame(props, geometry=geoms, crs=f"EPSG:{epsg}")
        return gdf.to_crs(WGS84) if int(epsg) != 4326 else gdf
    warnings.warn("No geometry present; returning attribute-only GeoDataFrame")
    return gpd.GeoDataFrame(props)


def download_as_geojson(service_url, service_name, output_dir=None):
    """Download an entire layer, save it as GeoJSON, and return (path, GeoDataFrame)."""
    output_dir = output_dir or OUTPUT_DIR
    print(f"\nDownloading: {service_name}")

    # Self-hosted ArcGIS Servers may need an IP-based token before they answer
    token = None
    if "arcgis.com" not in service_url.lower():
        token = try_get_server_token(service_url)
        if token:
            print("  Acquired server token")

    info = get_service_info(service_url, token=token)
    max_records = info.get("maxRecordCount", 1000)
    supports_pagination = info.get("advancedQueryCapabilities", {}).get("supportsPagination", False)
    oid_field = get_oid_field(info)

    capabilities = info.get("capabilities", "")
    if capabilities and "Query" not in capabilities:
        raise RuntimeError(
            f"Layer does not support Query operations (capabilities: '{capabilities}'). "
            "Verify the service URL or check if authentication is required."
        )

    # Field-based where clause; some servers reject the constant expression 1=1
    where_all = f"{oid_field} IS NOT NULL"
    token_param = {"token": token} if token else {}
    print(f"  maxRecordCount={max_records}, supportsPagination={supports_pagination}, oid={oid_field}")

    all_features, epsg, use_geojson = [], 4326, True
    rest_failed = False
    first_error = None

    try:
        use_offset = supports_pagination
        if use_offset:
            try:
                offset = 0
                while True:
                    features, epsg, use_geojson = query_page(
                        service_url,
                        {
                            "where": where_all,
                            "outFields": "*",
                            "resultOffset": offset,
                            "resultRecordCount": max_records,
                            "orderByFields": oid_field,
                            **token_param,
                        },
                        use_geojson,
                    )
                    if not features:
                        break
                    all_features.extend(features)
                    print(f"  {len(all_features)} features downloaded...")
                    if len(features) < max_records:
                        break
                    offset += max_records
            except RuntimeError as e:
                warnings.warn(f"Offset pagination failed ({e}), falling back to OID chunking")
                all_features, use_geojson, use_offset = [], True, False

        if not use_offset:
            oid_data = safe_get(
                f"{service_url}/query",
                {"where": where_all, "returnIdsOnly": "true", "f": "json", **token_param},
            )
            all_oids = sorted(oid_data.get("objectIds", []))
            print(f"  {len(all_oids)} total features, chunking by {max_records} OIDs")
            for i in range(0, len(all_oids), max_records):
                chunk = all_oids[i : i + max_records]
                features, epsg, use_geojson = query_page(
                    service_url,
                    {"objectIds": ",".join(str(o) for o in chunk), "outFields": "*", **token_param},
                    use_geojson,
                )
                all_features.extend(features)
                print(f"  {len(all_features)}/{len(all_oids)} features downloaded...")

    except Exception as e:
        rest_failed = True
        first_error = e
        warnings.warn(f"All REST queries failed ({e}), trying arcgis Python SDK")
        all_features = []

    # Only reach for the SDK if REST actually errored - an empty layer is not a failure
    if rest_failed:
        try:
            all_features = download_via_arcgis_sdk(service_url, max_records, where_all)
            epsg = 4326
        except Exception as e:
            raise RuntimeError(
                f"All download methods failed for {service_name}. "
                f"The service may require authentication or have query restrictions. "
                f"Last error: {e}"
            ) from first_error

    out_path = os.path.join(output_dir, f"{service_name}.geojson")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump({"type": "FeatureCollection", "features": all_features}, f)

    gdf = features_to_gdf(all_features, epsg)
    print(f"  Saved {len(all_features)} features -> {out_path}")
    return out_path, gdf

## 1. Download the jurisdiction services

Each source is saved to `OUTPUT_DIR` as GeoJSON and kept in `gdfs`. A source that
fails is reported and skipped rather than aborting the run.

Washoe County's FeatureServer currently answers every query with
`400 Unable to perform query operation`, so the notebook falls back to the
shapefile export at `WASHOE_SHAPEFILE`. Remove that fallback once the service is
fixed.

In [3]:
gdfs = {}
for name, url in SERVICE_URLS.items():
    try:
        _, gdfs[name] = download_as_geojson(url, name)
    except Exception as e:
        print(f"  ERROR - {name}: {e}")

# Fall back to the local Washoe export only if the service did not deliver
washoe = gdfs.get("Washoe_VHR")
if washoe is None or len(washoe) == 0:
    if os.path.exists(WASHOE_SHAPEFILE):
        gdfs["Washoe_VHR"] = gpd.read_file(WASHOE_SHAPEFILE)
        print(f"\nWashoe service unavailable - loaded {len(gdfs['Washoe_VHR'])} "
              f"features from {WASHOE_SHAPEFILE}")
    else:
        warnings.warn(f"Washoe service unavailable and {WASHOE_SHAPEFILE} not found")

print("\nDownload summary")
for name in SERVICE_URLS:
    gdf = gdfs.get(name)
    print(f"  {name:<24} {len(gdf) if gdf is not None else 'FAILED'}")
    if gdf is not None:
        print(f"    columns: {list(gdf.columns)}")


Downloading: ElDorado_VHR_Permits


c:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'gis.eldoradocounty.ca.gov'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


  maxRecordCount=2000, supportsPagination=True, oid=OBJECTID
  2000 features downloaded...
  2652 features downloaded...
  Saved 2652 features -> F:\GIS\PROJECTS\ResearchAnalysis\VHR\GeoJSON\ElDorado_VHR_Permits.geojson

Downloading: CSLT_VHR
  maxRecordCount=1000, supportsPagination=True, oid=objectid
  1000 features downloaded...
  1098 features downloaded...
  Saved 1098 features -> F:\GIS\PROJECTS\ResearchAnalysis\VHR\GeoJSON\CSLT_VHR.geojson

Downloading: Washoe_VHR


c:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'wcgisweb.washoecounty.us'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\amcclary\AppData\Local\Temp\ipykernel_36192\4042737116.py:118: UserWarning: GeoJSON query failed (Service error 400: Unable to complete operation. ['Unable to perform query operation.']), falling back to Esri JSON
  warnings.warn(f"GeoJSON query failed ({e}), falling back to Esri JSON")


  maxRecordCount=1000, supportsPagination=True, oid=entryId


C:\Users\amcclary\AppData\Local\Temp\ipykernel_36192\4042737116.py:244: UserWarning: Offset pagination failed (Service error 400: Unable to complete operation. ['Unable to perform query operation.']), falling back to OID chunking
  warnings.warn(f"Offset pagination failed ({e}), falling back to OID chunking")
C:\Users\amcclary\AppData\Local\Temp\ipykernel_36192\4042737116.py:267: UserWarning: All REST queries failed (Service error 400: Unable to complete operation. ['Unable to perform query operation.']), trying arcgis Python SDK
  warnings.warn(f"All REST queries failed ({e}), trying arcgis Python SDK")


  Trying arcgis Python SDK fallback...
  ERROR - Washoe_VHR: All download methods failed for Washoe_VHR. The service may require authentication or have query restrictions. Last error: {'error': {'code': 400, 'message': 'Unable to complete operation.', 'details': ['Unable to perform query operation.']}}

Downloading: Douglas_VHR_Occupancy


c:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'gisservices.douglasnv.us'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


  maxRecordCount=2000, supportsPagination=True, oid=OBJECTID
  556 features downloaded...
  Saved 556 features -> F:\GIS\PROJECTS\ResearchAnalysis\VHR\GeoJSON\Douglas_VHR_Occupancy.geojson

Downloading: Placer_VHR
  maxRecordCount=2000, supportsPagination=True, oid=OBJECTID
  2000 features downloaded...
  3369 features downloaded...
  Saved 3369 features -> F:\GIS\PROJECTS\ResearchAnalysis\VHR\GeoJSON\Placer_VHR.geojson

Washoe service unavailable - loaded 746 features from C:\Users\amcclary\Downloads\ShortTermRentals\ShortTermRentals.shp

Download summary
  ElDorado_VHR_Permits     2652
    columns: ['OBJECTID', 'License_Status', 'Account_Number', 'Number_of_Overnight_Guests', 'Contact_name', 'Contact_phone_number', 'Contact_Title', 'APN', 'Business_Address_', 'Business_Address_Pre_Direction', 'Business_Address_Street', 'Business_Address_Suffix', 'Business_Address_Post_Direction', 'Business_Address_Unit', 'Business_Address_City', 'Business_Address_State', 'Business_Address_Zip_Code', 

## 2. Normalize and combine

Each source names its fields differently, so `JURISDICTIONS` records the column
candidates and the APN format for each one.

Two things worth knowing:

- **APN formats differ by county.** El Dorado and CSLT use `###-###-###`, Washoe
  `###-###-##`, Douglas `####-##-###-###`, and Placer publishes a 12-digit APN
  whose trailing unit number is dropped to match `Parcel_Master`. Formatting them
  all the same way produces APNs that match nothing.
- **Field names drift.** Placer's columns carry the extract date
  (`PermittedSTRs_260508_APN`) and the Washoe shapefile truncates names to 10
  characters (`APPL_STATU`, `Max_Occupa`), so `find_col` falls back to a
  substring match and warns loudly when a field cannot be found.

Polygon sources are reduced to centroids, computed in UTM 10N rather than in
degrees, and everything is returned in WGS84.

In [4]:
"""Per-jurisdiction field specification.

Every source names its fields differently, and several of them change names each
fiscal year (Placer) or arrive truncated to 10 characters via shapefile export
(Washoe). Each entry therefore lists explicit column candidates first and falls
back to a normalized substring match.

apn_digits / apn_groups describe how the county writes an APN:

    El Dorado / CSLT   9 digits   ###-###-###
    Washoe             8 digits   ###-###-##
    Douglas           12 digits   ####-##-###-###
    Placer            12 digits   ###-###-###  (trailing unit number dropped,
                                                matching Parcel_Master)
"""

JURISDICTIONS = {
    "El Dorado County": {
        "key": "ElDorado_VHR_Permits",
        "apn": ["APN"],
        "status": ["License_Status"],
        "occupancy": ["Number_of_Overnight_Guests"],
        "apn_digits": 9,
        "apn_groups": (3, 3, 3),
    },
    "City of South Lake Tahoe": {
        "key": "CSLT_VHR",
        "apn": ["apn", "prcl_id"],
        "status": ["status"],
        "occupancy": ["occupancy", "occpy_max"],
        "apn_digits": 9,
        "apn_groups": (3, 3, 3),
    },
    "Washoe County": {
        # B1_PARCEL_NBR is preferred over APN: the APN field is numeric and has
        # lost leading zeros on 41 of 746 records.
        "key": "Washoe_VHR",
        "apn": ["B1_PARCEL_NBR", "B1_PARCEL_", "APN"],
        "status": ["APPL_STATUS", "APPL_STATU"],
        "occupancy": ["Max_Occupancy", "Max_Occupa"],
        "apn_digits": 8,
        "apn_groups": (3, 3, 2),
    },
    "Douglas County": {
        # APN_1 is the pre-formatted string; APN is a 12-digit number.
        "key": "Douglas_VHR_Occupancy",
        "apn": ["APN_1", "APN"],
        "status": ["Permit_Status"],
        "occupancy": ["Max_Nighttime_Occupancy"],
        "apn_digits": 12,
        "apn_groups": (4, 2, 3, 3),
    },
    "Placer County": {
        # Field names carry the extract date, so the fuzzy match does the work.
        "key": "Placer_VHR",
        "apn": [],
        "status": [],
        "occupancy": [],
        "apn_digits": 12,
        "apn_groups": (3, 3, 3),
        # The service publishes permitted properties only - it carries neither
        # a status nor an occupancy field, so don't warn about them.
        "optional": ("status", "occupancy"),
    },
}

# Fuzzy fallbacks, tried in order after the explicit candidates above
APN_TOKENS       = ("apn", "parcelnbr", "parcelnumber", "parcelid")
STATUS_TOKENS    = ("status", "statu")
OCCUPANCY_TOKENS = ("occupancy", "occup", "guests")

# Jurisdictions that publish only permitted properties, so every record is active
ASSUME_ACTIVE = {"Placer County"}


def _norm(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


def find_col(gdf, candidates=(), tokens=()):
    """Exact (case/punctuation-insensitive) match first, then substring match."""
    lookup = {_norm(c): c for c in gdf.columns}
    for cand in candidates:
        if _norm(cand) in lookup:
            return lookup[_norm(cand)]
    for token in tokens:
        for key, col in lookup.items():
            if token in key:
                return col
    return None


def format_apn(value, groups, expected_digits):
    """Format an APN into the county's own punctuation, or None if unusable."""
    try:
        if value is None or pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(value, float) and float(value).is_integer():
        value = int(value)
    digits = re.sub(r"\D", "", str(value))
    if not digits:
        return None
    if len(digits) < expected_digits:
        digits = digits.zfill(expected_digits)   # numeric fields drop leading zeros
    keep = sum(groups)
    if len(digits) < keep:
        return None
    parts, i = [], 0
    for g in groups:
        parts.append(digits[i:i + g])
        i += g
    return "-".join(parts)


def _string_col(gdf, col, n):
    """Return a stripped string Series, blanks as NA, or an all-NA Series."""
    if col is None:
        return pd.Series(pd.NA, index=range(n), dtype="string")
    s = pd.Series(gdf[col].to_numpy(), dtype="object").astype("string").str.strip()
    return s.replace("", pd.NA)


def to_wgs84_points(gdf):
    """Reduce polygons to centroids and return the layer in WGS84.

    Centroids are computed in a projected CRS - taking them in degrees warns and
    puts the point in the wrong place.
    """
    if gdf.crs is None:
        gdf = gdf.set_crs(WGS84)
    is_poly = gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
    if is_poly.any():
        gdf = gdf.to_crs(PROJECTED_CRS)
        gdf.loc[is_poly, "geometry"] = gdf.loc[is_poly, "geometry"].centroid
    return gdf.to_crs(WGS84)


def extract_fields(gdf, jurisdiction, spec):
    """Reduce a source layer to apn / status / max_occupancy / jurisdiction + point."""
    gdf = gdf.reset_index(drop=True)
    n = len(gdf)

    apn_col  = find_col(gdf, spec["apn"], APN_TOKENS)
    stat_col = find_col(gdf, spec["status"], STATUS_TOKENS)
    occ_col  = find_col(gdf, spec["occupancy"], OCCUPANCY_TOKENS)
    print(f"  {jurisdiction:<26} apn={apn_col}  status={stat_col}  occupancy={occ_col}")
    optional = spec.get("optional", ())
    for label, col in (("apn", apn_col), ("status", stat_col), ("occupancy", occ_col)):
        if col is None and label not in optional:
            warnings.warn(f"{jurisdiction}: no {label} column found - values will be null")

    if apn_col is None:
        apn = pd.Series(pd.NA, index=range(n), dtype="string")
    else:
        raw = gdf[apn_col].to_numpy()
        apn = pd.Series(
            [format_apn(v, spec["apn_groups"], spec["apn_digits"]) for v in raw],
            dtype="string",
        )
        bad = int(apn.isna().sum())
        if bad:
            print(f"    {bad} of {n} records have no usable APN")

    out = gpd.GeoDataFrame(
        {
            "apn": apn,
            "status": _string_col(gdf, stat_col, n),
            "max_occupancy": pd.to_numeric(
                gdf[occ_col], errors="coerce"
            ).astype("Int64").reset_index(drop=True) if occ_col else pd.Series(
                pd.NA, index=range(n), dtype="Int64"
            ),
            "jurisdiction": pd.Series([jurisdiction] * n, dtype="string"),
        },
        geometry=gdf.geometry.to_numpy(),
        crs=gdf.crs,
    )
    return to_wgs84_points(out)


parts = []
print("Field detection")
for jurisdiction, spec in JURISDICTIONS.items():
    gdf = gdfs.get(spec["key"])
    if gdf is None or len(gdf) == 0:
        warnings.warn(f"{jurisdiction}: no data loaded - skipped")
        continue
    parts.append(extract_fields(gdf, jurisdiction, spec))

if not parts:
    raise RuntimeError("No jurisdictions loaded - nothing to combine")

gdf_combined = gpd.GeoDataFrame(
    pd.concat(parts, ignore_index=True), geometry="geometry", crs=WGS84
)

print(f"\nCombined: {len(gdf_combined)} records")
print(gdf_combined.groupby("jurisdiction", dropna=False).size().rename("count").to_string())

# El Dorado publishes one row per contact, so an APN can legitimately repeat.
dupes = gdf_combined.duplicated(["apn", "jurisdiction"]) & gdf_combined["apn"].notna()
print(f"\n{int(dupes.sum())} records share an APN with an earlier record "
      f"(kept - review before any APN-based join)")

Field detection
  El Dorado County           apn=APN  status=License_Status  occupancy=Number_of_Overnight_Guests
    44 of 2652 records have no usable APN
  City of South Lake Tahoe   apn=apn  status=status  occupancy=occupancy
  Washoe County              apn=B1_PARCEL_  status=APPL_STATU  occupancy=Max_Occupa
  Douglas County             apn=APN_1  status=Permit_Status  occupancy=Max_Nighttime_Occupancy
  Placer County              apn=PermittedSTRs_260508_APN  status=None  occupancy=None

Combined: 8421 records
jurisdiction
City of South Lake Tahoe    1098
Douglas County               556
El Dorado County            2652
Placer County               3369
Washoe County                746

791 records share an APN with an earlier record (kept - review before any APN-based join)


## 3. Standardize permit status

Each jurisdiction uses its own permit vocabulary, so `vhr_types_lookup.csv` maps
every `(status, jurisdiction)` pair to one of Active / Inactive / Pending. That
file is maintained by hand; this notebook only reports what needs adding to it.

### 3a. Export the status values seen this run

Writes `vhr_types.csv` and lists any status/jurisdiction pair that
`vhr_types_lookup.csv` does not cover yet. Add a `status_simple` for each of
those before running 3b.

In [5]:
STATUS_BLANK = "(blank)"   # merge sentinel so missing statuses join like any other value


def status_key(series):
    return series.astype("string").str.strip().replace("", pd.NA).fillna(STATUS_BLANK)


vhr_types = (
    gdf_combined[["status", "jurisdiction"]]
    .drop_duplicates()
    .sort_values(["jurisdiction", "status"], na_position="first")
    .reset_index(drop=True)
)
vhr_types.to_csv(VHR_TYPES_CSV, index=False)
print(f"{len(vhr_types)} status / jurisdiction pairs -> {VHR_TYPES_CSV}\n")
print(vhr_types.fillna(STATUS_BLANK).to_string(index=False))

# Show which pairs the hand-maintained crosswalk does not cover yet
if os.path.exists(VHR_TYPES_LOOKUP_CSV):
    _known = pd.read_csv(VHR_TYPES_LOOKUP_CSV, dtype="string")
    _have = set(zip(status_key(_known["status"]), _known["jurisdiction"].str.strip()))
    _need = set(zip(status_key(vhr_types["status"]), vhr_types["jurisdiction"]))
    _missing = sorted(_need - _have)
    if _missing:
        print(f"\nNOT YET IN {os.path.basename(VHR_TYPES_LOOKUP_CSV)} - add a "
              f"status_simple for each before running the next cell:")
        for status, jurisdiction in _missing:
            print(f"  {status}  |  {jurisdiction}")
    else:
        print(f"\nAll pairs are covered by {os.path.basename(VHR_TYPES_LOOKUP_CSV)}.")
else:
    print(f"\n{VHR_TYPES_LOOKUP_CSV} does not exist yet - copy vhr_types.csv, add a "
          f"status_simple column, and save it under that name.")

21 status / jurisdiction pairs -> c:\Users\amcclary\Documents\GitHub\ParcelUpdate\vhr_types.csv

                          status             jurisdiction
                          issued City of South Lake Tahoe
                         Current           Douglas County
                         pending           Douglas County
                         (blank)         El Dorado County
              HHR Permit- Active         El Dorado County
HHR Permit- Approved for Payment         El Dorado County
              HHR Permit- Closed         El Dorado County
            HHR Permit- Complete         El Dorado County
           HHR Permit- Submitted         El Dorado County
             VHR Permit - Active         El Dorado County
VHR Permit- Approved for Payment         El Dorado County
              VHR Permit- Closed         El Dorado County
            VHR Permit- Complete         El Dorado County
          VHR Permit- Incomplete         El Dorado County
             VHR Permit- Revoked 

### 3b. Apply the status crosswalk

Merges `status_simple` onto the combined layer. Blank statuses join through a
`(blank)` sentinel so they behave like any other value, and the merge is
validated many-to-one so a duplicated crosswalk row cannot silently multiply
records.

In [6]:
if not os.path.exists(VHR_TYPES_LOOKUP_CSV):
    raise FileNotFoundError(
        f"{VHR_TYPES_LOOKUP_CSV} not found. Run the cell above, then copy "
        f"vhr_types.csv to vhr_types_lookup.csv and fill in a status_simple column."
    )

vhr_types_lookup = pd.read_csv(VHR_TYPES_LOOKUP_CSV, dtype="string")
vhr_types_lookup["jurisdiction"]  = vhr_types_lookup["jurisdiction"].str.strip()
vhr_types_lookup["status_simple"] = vhr_types_lookup["status_simple"].str.strip()
vhr_types_lookup["_status_key"]   = status_key(vhr_types_lookup["status"])

dupes = vhr_types_lookup.duplicated(["_status_key", "jurisdiction"])
if dupes.any():
    raise ValueError(
        "Duplicate status/jurisdiction rows in the crosswalk - a merge would "
        f"multiply records:\n{vhr_types_lookup.loc[dupes, ['status', 'jurisdiction']].to_string(index=False)}"
    )

gdf_combined = gdf_combined.drop(columns=["status_simple"], errors="ignore")
gdf_combined["_status_key"] = status_key(gdf_combined["status"])
gdf_combined = gdf_combined.merge(
    vhr_types_lookup[["_status_key", "jurisdiction", "status_simple"]],
    on=["_status_key", "jurisdiction"],
    how="left",
    validate="m:1",
).drop(columns="_status_key")

# These jurisdictions publish permitted properties only
gdf_combined.loc[gdf_combined["jurisdiction"].isin(ASSUME_ACTIVE), "status_simple"] = "Active"

unmapped = (
    gdf_combined.loc[gdf_combined["status_simple"].isna(), ["status", "jurisdiction"]]
    .drop_duplicates()
)
if len(unmapped):
    warnings.warn(f"{len(unmapped)} status/jurisdiction pairs have no status_simple")
    print("Unmapped pairs (add them to vhr_types_lookup.csv):")
    print(unmapped.fillna(STATUS_BLANK).to_string(index=False))

print(gdf_combined["status_simple"].value_counts(dropna=False).to_string())

status_simple
Active      7077
Inactive    1054
Pending      290


## 4. Flag records inside the Basin

In [7]:
_, gdf_basin = download_as_geojson(BASIN_URL, "TRPA_Basin")

if gdf_basin is None or len(gdf_basin) == 0 or not gdf_basin.geometry.notna().any():
    raise RuntimeError(f"Basin boundary not loaded from {BASIN_URL}")

if gdf_basin.crs is None:
    gdf_basin = gdf_basin.set_crs(WGS84)
basin_geom = gdf_basin.to_crs(WGS84).geometry.union_all()

gdf_combined["in_basin"] = gdf_combined.geometry.within(basin_geom)
print(gdf_combined["in_basin"].value_counts().to_string())
print()
print(
    pd.crosstab(gdf_combined["jurisdiction"], gdf_combined["in_basin"]).to_string()
)


Downloading: TRPA_Basin


c:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'maps.trpa.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


  maxRecordCount=2000, supportsPagination=True, oid=OBJECTID
  1 features downloaded...
  Saved 1 features -> F:\GIS\PROJECTS\ResearchAnalysis\VHR\GeoJSON\TRPA_Basin.geojson
in_basin
True     6730
False    1691

in_basin                  False  True 
jurisdiction                          
City of South Lake Tahoe      0   1098
Douglas County              121    435
El Dorado County            497   2155
Placer County              1024   2345
Washoe County                49    697


## 5. Keep only Active permits inside the Basin

Everything downstream - the shapefile, the GeoJSON, and both feature classes -
describes `gdf_vhr`, the Active in-Basin subset. `gdf_combined` is left intact so
the status and Basin counts above can still be reviewed.

A record is dropped if `status_simple` is anything other than `Active` (a status
the crosswalk does not map yet included) or if its point falls outside the TRPA
boundary.


In [8]:
ACTIVE_STATUS = "Active"

is_active = gdf_combined["status_simple"].eq(ACTIVE_STATUS).fillna(False).astype(bool)
in_basin = gdf_combined["in_basin"].fillna(False).astype(bool)

gdf_vhr = gdf_combined[is_active & in_basin].reset_index(drop=True)

print(f"{len(gdf_combined)} records downloaded")
print(f"  {int(is_active.sum()):>6} active")
print(f"  {int(in_basin.sum()):>6} in basin")
print(f"  {len(gdf_vhr):>6} active and in basin - kept")

if gdf_vhr.empty:
    raise RuntimeError(
        "No Active in-Basin records - check the status crosswalk before exporting"
    )

print()
print(gdf_vhr.groupby("jurisdiction", dropna=False).size().rename("kept").to_string())


8421 records downloaded
    7077 active
    6730 in basin
    5658 active and in basin - kept

jurisdiction
City of South Lake Tahoe    1098
Douglas County               433
El Dorado County            1085
Placer County               2345
Washoe County                697


## 6. Save the combined layer

Written once, after every field exists. Only the Active in-Basin records
(`gdf_vhr`) are written. Extension dtypes (`string`, `Int64`,
`boolean`) are cast to plain types first because neither the shapefile writer nor
`to_featureclass` accepts them.

In [9]:
def for_export(gdf):
    """Cast pandas extension dtypes to types every GDAL/arcpy writer accepts."""
    out = gdf.copy()
    for col in out.columns:
        if col == out.geometry.name:
            continue
        dtype = str(out[col].dtype)
        if dtype == "string":
            out[col] = out[col].astype(object).where(out[col].notna(), None)
        elif dtype.startswith(("Int", "UInt")):
            out[col] = out[col].astype("float64")   # keeps nulls representable
        elif dtype == "boolean" or out[col].dtype == bool:
            out[col] = out[col].map({True: 1, False: 0}).fillna(-1).astype("int32")
    return out


gdf_export = for_export(gdf_vhr)

# Shapefile: kept for backwards compatibility. DBF truncates names to 10
# characters (max_occupancy -> max_occupa, jurisdiction -> jurisdicti,
# status_simple -> status_sim), so the GeoJSON below is the readable copy.
combined_shp = os.path.join(OUTPUT_DIR, "VHR_Combined.shp")
combined_geojson = os.path.join(OUTPUT_DIR, "VHR_Combined.geojson")

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*(truncated|Normalized/laundered).*")
    gdf_export.to_file(combined_shp)
gdf_export.to_file(combined_geojson, driver="GeoJSON")

print(f"Saved {len(gdf_export)} records")
print(f"  {combined_shp}")
print(f"  {combined_geojson}")
print("\nFields:", list(gdf_vhr.columns))
print(gdf_vhr.head().to_string())

Saved 5658 records
  F:\GIS\PROJECTS\ResearchAnalysis\VHR\GeoJSON\VHR_Combined.shp
  F:\GIS\PROJECTS\ResearchAnalysis\VHR\GeoJSON\VHR_Combined.geojson

Fields: ['apn', 'status', 'max_occupancy', 'jurisdiction', 'geometry', 'status_simple', 'in_basin']
           apn               status  max_occupancy      jurisdiction                     geometry status_simple  in_basin
0  015-034-017   HHR Permit- Active              4  El Dorado County   POINT (-120.1302 39.06711)        Active      True
1  015-351-004  VHR Permit - Active              4  El Dorado County    POINT (-120.1258 39.0671)        Active      True
2  015-351-004  VHR Permit - Active              4  El Dorado County    POINT (-120.1258 39.0671)        Active      True
3  014-292-011  VHR Permit - Active              6  El Dorado County  POINT (-120.13695 39.06714)        Active      True
4  015-381-007  VHR Permit - Active              6  El Dorado County  POINT (-120.12749 39.06692)        Active      True


## 7. Load to the staging FGDB and join to Parcel_Master

Two joined outputs are produced from the same Active in-Basin records:

- `VHR_Parcel_Join` - the VHR point carrying the `Parcel_Master` attributes
- `VHR_Parcel_Polygon` - the same records on the `Parcel_Master` polygon, so a
  permit can be mapped as a parcel rather than as a point

The two are row for row: one record per permit, the same attributes, the same
feature count. The polygon output is written in `Parcel_Master`'s own
coordinate system, not the WGS84 of the points, so the parcel shapes are exact
copies rather than reprojected ones - crossing the datum moves every parcel a
metre or two off the source. Each permit takes a single parcel polygon, so a permit standing
on overlapping parcels is not written twice, and a permit that falls on no
parcel keeps its attributes and is written with no shape rather than dropped.
The point output already treats an unmatched permit that way, with null parcel
fields.

Requires the SDE connections under `F:\GIS\DB_CONNECT`.

In [10]:
from arcgis.features import GeoAccessor

for path in (WORKSPACE, PARCEL_MASTER, PARCEL_VHR):
    if not arcpy.Exists(path):
        raise RuntimeError(f"Not reachable - check the SDE connection: {path}")

# 1. Write the combined points to the staging geodatabase
if arcpy.Exists(VHR_FC):
    arcpy.management.Delete(VHR_FC)
sdf = GeoAccessor.from_geodataframe(for_export(gdf_vhr), column_name="SHAPE")
sdf.spatial.to_featureclass(location=VHR_FC, sanitize_columns=False)
print(f"Created: {VHR_FC}  ({arcpy.management.GetCount(VHR_FC)[0]} features)")

# 2. Carry over only the Parcel_Master fields that SDE.Parcel_VHR already defines
vhr_schema = {f.name.split(".")[-1].upper() for f in arcpy.ListFields(PARCEL_VHR)}
pm_join_fields = [
    f for f in arcpy.ListFields(PARCEL_MASTER)
    if not f.required and f.name.split(".")[-1].upper() in vhr_schema
]
print(f"\nParcel_Master fields matching the VHR schema ({len(pm_join_fields)}):")
for f in pm_join_fields:
    print(f"  {f.name}  ({f.type})")

fm = arcpy.FieldMappings()
fm.addTable(VHR_FC)
for field in pm_join_fields:
    fmap = arcpy.FieldMap()
    fmap.addInputField(PARCEL_MASTER, field.name)
    fm.addFieldMap(fmap)

# 3. Spatial join - VHR points within Parcel_Master polygons
if arcpy.Exists(VHR_PARCEL_FC):
    arcpy.management.Delete(VHR_PARCEL_FC)
arcpy.analysis.SpatialJoin(
    target_features=VHR_FC,
    join_features=PARCEL_MASTER,
    out_feature_class=VHR_PARCEL_FC,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    field_mapping=fm,
    match_option="WITHIN",
)

for fname in ("Join_Count", "TARGET_FID"):
    if arcpy.ListFields(VHR_PARCEL_FC, fname):
        arcpy.management.DeleteField(VHR_PARCEL_FC, fname)

print(f"\nCreated: {VHR_PARCEL_FC}  ({arcpy.management.GetCount(VHR_PARCEL_FC)[0]} features)")
print("Output fields:", [f.name for f in arcpy.ListFields(VHR_PARCEL_FC) if not f.required])


# 4. The same records with the parcel polygon in place of the point. Built row
#    for row from the point join output above, so the two feature classes hold
#    the same attributes and the same number of features - a spatial join with
#    Parcel_Master as the target cannot promise either, since it drops a permit
#    that matched no parcel and writes one twice where parcels overlap.
from shapely import wkb as shapely_wkb

# The parcel shapes are copied out exactly as Parcel_Master holds them, in its own
# coordinate system and with no reprojection, so the output lines up with the
# source. Writing them in the WGS84 of the point layer instead crosses a datum and
# leaves every parcel a metre or two off.
parcel_sr = arcpy.Describe(PARCEL_MASTER).spatialReference
parcel_crs = f"EPSG:{parcel_sr.factoryCode}" if parcel_sr.factoryCode else None
print(f"Parcel polygons written in {parcel_sr.name}")

# Only the parcels holding a permit are read.
PARCEL_LYR = "parcel_master_vhr_lyr"
if arcpy.Exists(PARCEL_LYR):
    arcpy.management.Delete(PARCEL_LYR)
arcpy.management.MakeFeatureLayer(PARCEL_MASTER, PARCEL_LYR)
arcpy.management.SelectLayerByLocation(PARCEL_LYR, "CONTAINS", VHR_FC)
print(f"\n{arcpy.management.GetCount(PARCEL_LYR)[0]} parcels contain at least one VHR point")

parcel_shapes = {}                  # parcel OID -> arcpy polygon, copied to the output
parcel_oids, parcel_geoms = [], []  # the same parcels as shapely, for the lookup
with arcpy.da.SearchCursor(PARCEL_LYR, ["OID@", "SHAPE@", "SHAPE@WKB"]) as cur:
    for oid, shp, wkb in cur:
        if shp is None or wkb is None:
            continue
        parcel_shapes[oid] = shp
        parcel_oids.append(oid)
        parcel_geoms.append(shapely_wkb.loads(bytes(wkb)))
parcels = gpd.GeoDataFrame(
    {"parcel_oid": parcel_oids}, geometry=parcel_geoms, crs=parcel_crs
)

# Read the joined points once - every attribute the polygon output needs is
# already on them, so nothing has to be field mapped a second time. Only the
# points are moved into the parcel coordinate system, and only so the two can be
# compared; no point geometry is written out here.
poly_fields = [f.name for f in arcpy.ListFields(VHR_PARCEL_FC) if not f.required]
attrs, points = [], []
with arcpy.da.SearchCursor(
    VHR_PARCEL_FC, poly_fields + ["SHAPE@WKB"], spatial_reference=parcel_sr
) as cur:
    for row in cur:
        attrs.append(row[:-1])
        points.append(shapely_wkb.loads(bytes(row[-1])) if row[-1] else None)

# One parcel per permit: where parcels overlap a point falls in more than one,
# so only the first match is kept.
pts = gpd.GeoDataFrame({"row": range(len(points))}, geometry=points, crs=parcel_crs)
matched = gpd.sjoin(
    pts[pts.geometry.notna()], parcels, how="inner", predicate="within"
)
matched = matched[~matched["row"].duplicated()]
row_parcel = dict(zip(matched["row"], matched["parcel_oid"]))

if arcpy.Exists(VHR_PARCEL_POLY_FC):
    arcpy.management.Delete(VHR_PARCEL_POLY_FC)
arcpy.management.CreateFeatureclass(
    out_path=WORKSPACE,
    out_name=os.path.basename(VHR_PARCEL_POLY_FC),
    geometry_type="POLYGON",
    template=VHR_PARCEL_FC,
    spatial_reference=parcel_sr,
)
with arcpy.da.InsertCursor(VHR_PARCEL_POLY_FC, poly_fields + ["SHAPE@"]) as cur:
    for i, values in enumerate(attrs):
        cur.insertRow(tuple(values) + (parcel_shapes.get(row_parcel.get(i)),))
arcpy.management.Delete(PARCEL_LYR)

point_count = int(arcpy.management.GetCount(VHR_PARCEL_FC)[0])
poly_count = int(arcpy.management.GetCount(VHR_PARCEL_POLY_FC)[0])
print(f"\nCreated: {VHR_PARCEL_POLY_FC}  ({poly_count} features)")
print(f"  {point_count} features in {os.path.basename(VHR_PARCEL_FC)}")
if poly_count != point_count:
    raise RuntimeError("Polygon and point outputs disagree - the two must stay row for row")
no_parcel = len(attrs) - len(row_parcel)
if no_parcel:
    print(f"  {no_parcel} permits fell on no parcel and were written with no shape")
print("Output fields:", [f.name for f in arcpy.ListFields(VHR_PARCEL_POLY_FC) if not f.required])


Created: F:\GIS\PROJECTS\ResearchAnalysis\VHR\Data\VHR_Staging.gdb\VHR_Combined  (5658 features)

Parcel_Master fields matching the VHR schema (72):
  APN  (String)
  PPNO  (Double)
  HSE_NUMBR  (String)
  UNIT_NUMBR  (String)
  STR_DIR  (String)
  STR_NAME  (String)
  STR_SUFFIX  (String)
  APO_ADDRESS  (String)
  PSTL_TOWN  (String)
  PSTL_STATE  (String)
  PSTL_ZIP5  (String)
  OWN_FIRST  (String)
  OWN_LAST  (String)
  OWN_FULL  (String)
  MAIL_ADD1  (String)
  MAIL_ADD2  (String)
  MAIL_CITY  (String)
  MAIL_STATE  (String)
  MAIL_ZIP5  (String)
  JURISDICTION  (String)
  COUNTY  (String)
  OWNERSHIP_TYPE  (String)
  COUNTY_LANDUSE_CODE  (String)
  COUNTY_LANDUSE_DESCRIPTION  (String)
  EXISTING_LANDUSE  (String)
  REGIONAL_LANDUSE  (String)
  IPES_SCORE  (Double)
  AS_LANDVALUE  (Integer)
  AS_IMPROVALUE  (Integer)
  AS_SUM  (Integer)
  TAX_LANDVALUE  (Integer)
  TAX_IMPROVALUE  (Integer)
  TAX_SUM  (Integer)
  UNITS  (String)
  BEDROOMS  (String)
  BATHROOMS  (String)
  BUILDING